# 10merra2: 23 haze seasons of NASA MERRA-2 aerosol

Part C of PF5. Downloads daily means of smoke, PM2.5 parts and smoke flux for every August to October from 2003 to 2026, for the study region, from NASA's MERRA-2 reanalysis.

**Why MERRA-2 and not GEOS-FP.** GEOS-FP changes version every few years; MERRA-2 is one fixed model run from 1980 to today, so 2003 and 2025 are directly comparable, which is what a climatology needs. MERRA-2 has no separate brown carbon; its organic carbon includes fire smoke, so "smoke" here is organic + black carbon (`notebook 11` uses the fire regions and the Pacific corridor, where that is mostly fire smoke).

**How it downloads.** NASA's OPeNDAP service cuts the region and the variables out on the server, so only a small piece of each daily file travels. Each day's 24 hourly fields are thinned to 4 (every 6 hours) and averaged on your PC into one small daily file in `data/raw/merra2/`. Five days are fetched at the same time (NASA asks for no more than 5 connections). Finished days are never fetched again.

Two ways are tried for each day: **fast** (the server packs the whole cut-out into one compressed netCDF file: one request per day) and, if that fails, **standard** (one request per variable: about 25 s per day in the first test). In test mode the first day is fetched both ways and compared, so we know the fast way gives identical numbers.

**Try it first.** With `TEST_MODE = True` (Section 0) the notebook fetches only a few days and prints the expected total time. Then set `TEST_MODE = False` and run again for everything. The full run is about 2,200 days; plan it as an overnight run if the test says more than an hour.

## Section 0: settings

In [6]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "data" / "raw" / "merra2"

TEST_MODE = False  # <- True: fetch TEST_DAYS days and estimate the total time; False: fetch everything
TEST_DAYS = 6  # <-
FIRST_YEAR = 2003  # <-
LAST_YEAR = 2026  # <- MERRA-2 runs about 3 to 4 weeks behind; missing recent days are simply skipped
SEASON_MONTHS = [8, 9, 10]  # <- August to October

BOX_WEST = 90.0  # <-
BOX_EAST = 160.0  # <-
BOX_SOUTH = -15.0  # <-
BOX_NORTH = 30.0  # <- 30N covers the typhoon box (5N to 25N) with margin

HOUR_STEP = 6  # <- every 6th hourly field (4 per day)
WORKERS = 5  # <- days fetched at the same time (NASA asks for no more than 5)
TRIES = 4  # <- attempts per day

VARIABLES = ["OCSMASS", "BCSMASS", "SSSMASS25", "DUSMASS25", "SO4SMASS", "TOTEXTTAU", "OCCMASS", "BCCMASS", "OCFLUXU", "OCFLUXV", "BCFLUXU", "BCFLUXV"]  # <-

## Section 1: imports, login and the list of days

In [7]:
import time
import urllib.parse
import threading
import warnings
import concurrent.futures as cf
import numpy as np
import pandas as pd
import xarray as xr
import earthaccess

warnings.filterwarnings("ignore")
OUT_DIR.mkdir(parents=True, exist_ok=True)
auth = earthaccess.login(persist=True)
print("Earthdata login:", auth.authenticated)

days = pd.DatetimeIndex([d for y in range(FIRST_YEAR, LAST_YEAR + 1) for d in pd.date_range(f"{y}-{SEASON_MONTHS[0]:02d}-01", pd.Timestamp(f"{y}-{SEASON_MONTHS[-1]:02d}-01") + pd.offsets.MonthEnd(0))])
todo = [d for d in days if not (OUT_DIR / f"merra2_{d:%Y%m%d}.nc").exists()]
print(f"Season days {FIRST_YEAR} to {LAST_YEAR}: {len(days)}; already downloaded: {len(days) - len(todo)}; to do: {len(todo)}")

Earthdata login: True
Season days 2003 to 2026: 2208; already downloaded: 12; to do: 2196


In [8]:
# Find the OPeNDAP address of each day's file through NASA's search (one query per season)
def opendap_urls(year):
    res = earthaccess.search_data(short_name="M2T1NXAER", temporal=(f"{year}-{SEASON_MONTHS[0]:02d}-01", f"{year}-{SEASON_MONTHS[-1]:02d}-31"))
    out = {}
    for g in res:
        day = pd.Timestamp(g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10])
        links = [u["URL"] for u in g["umm"].get("RelatedUrls", []) if "opendap" in u["URL"].lower()]
        cloud = [u for u in links if "opendap.earthdata.nasa.gov" in u]
        data = [u for u in g.data_links() if u.endswith(".nc4")]
        out[day] = {"cloud": cloud[0] if cloud else None, "data": data[0] if data else None}
    return out


years_needed = sorted(set(d.year for d in todo))
if TEST_MODE:
    years_needed = years_needed[:1]
URLS = {}
for y in years_needed:
    URLS.update(opendap_urls(y))
    print(y, ":", len([d for d in URLS if d.year == y]), "daily files found")
example = next(iter(URLS.values())) if URLS else None
print("Example addresses:", example)

2003 : 92 daily files found
2004 : 92 daily files found
2005 : 92 daily files found
2006 : 92 daily files found
2007 : 92 daily files found
2008 : 92 daily files found
2009 : 92 daily files found
2010 : 92 daily files found
2011 : 92 daily files found
2012 : 92 daily files found
2013 : 92 daily files found
2014 : 92 daily files found
2015 : 92 daily files found
2016 : 92 daily files found
2017 : 92 daily files found
2018 : 92 daily files found
2019 : 92 daily files found
2020 : 92 daily files found
2021 : 92 daily files found
2022 : 92 daily files found
2023 : 92 daily files found
2024 : 92 daily files found
2025 : 92 daily files found
2026 : 62 daily files found
Example addresses: {'cloud': 'https://opendap.earthdata.nasa.gov/collections/C1276812830-GES_DISC/granules/M2T1NXAER.5.12.4%3AMERRA2_300.tavg1_2d_aer_Nx.20030801.nc4', 'data': 'https://data.gesdisc.earthdata.nasa.gov/data/MERRA2/M2T1NXAER.5.12.4/2003/08/MERRA2_300.tavg1_2d_aer_Nx.20030801.nc4'}


## Section 2: download

For each day: open the file through OPeNDAP (NASA's cloud OPeNDAP first, the GES DISC server as a backup), cut out the region and variables, thin to every 6th hour, average, and save `merra2_YYYYMMDD.nc` (about 0.5 MB).

In [9]:
LAT_ALL = np.arange(-90, 90.01, 0.5)
LON_ALL = np.arange(-180, 179.99, 0.625)
J0 = int(np.abs(LAT_ALL - BOX_SOUTH).argmin())
J1 = int(np.abs(LAT_ALL - BOX_NORTH).argmin())
I0 = int(np.abs(LON_ALL - BOX_WEST).argmin())
I1 = int(np.abs(LON_ALL - BOX_EAST).argmin())
_local = threading.local()


def session():
    if not hasattr(_local, "s"):
        _local.s = earthaccess.get_requests_https_session()
    return _local.s


def candidate_urls(info):
    urls = []
    if info.get("cloud"):
        ce = "?dap4.ce=" + ";".join(["/time", "/lat", "/lon"] + [f"/{v}" for v in VARIABLES])
        urls.append(info["cloud"].replace("https://", "dap4://") + ce)
    if info.get("data"):
        gd = info["data"].replace("https://data.gesdisc.earthdata.nasa.gov/data/", "https://goldsmr4.gesdisc.eosdis.nasa.gov/opendap/")
        urls.append(gd.replace("https://", "dap2://"))
    return urls


def get_fast(info, day):
    # One request: the server returns the cut-out as a compressed netCDF-4 file
    if not info.get("cloud"):
        raise ValueError("no cloud OPeNDAP address")
    t_idx = f"[0:{HOUR_STEP}:23]"
    parts = ["/time" + t_idx, f"/lat[{J0}:{J1}]", f"/lon[{I0}:{I1}]"] + [f"/{v}{t_idx}[{J0}:{J1}][{I0}:{I1}]" for v in VARIABLES]
    ce = urllib.parse.quote(";".join(parts), safe="/")
    url = info["cloud"] + ".dap.nc4?dap4.ce=" + ce
    r = session().get(url, timeout=180)
    r.raise_for_status()
    if not r.content[:4] == b"\x89HDF":
        raise ValueError("answer is not a netCDF-4 file: " + r.text[:80])
    tmp = OUT_DIR / f"_fast_{day:%Y%m%d}_{threading.get_ident()}.nc4"
    tmp.write_bytes(r.content)
    with xr.open_dataset(tmp) as ds:
        sub = ds[VARIABLES].load()
    tmp.unlink()
    return sub


def get_standard(info, day):
    # One request per variable through pydap
    last = None
    for url in candidate_urls(info):
        try:
            ds = xr.open_dataset(url, engine="pydap", session=session())
            return ds[VARIABLES].isel(time=slice(0, 24, HOUR_STEP), lat=slice(J0, J1 + 1), lon=slice(I0, I1 + 1)).load()
        except Exception as e:
            last = e
    raise last


def save_daily(sub, day, out_file):
    daily = sub.mean("time", keep_attrs=True).astype("float32").expand_dims(time=[day])
    tmp = out_file.with_suffix(".tmp")
    daily.to_netcdf(tmp)
    tmp.replace(out_file)


def fetch_day(day):
    out_file = OUT_DIR / f"merra2_{day:%Y%m%d}.nc"
    if out_file.exists():
        return day, "skip", 0.0
    info = URLS.get(day)
    if info is None:
        return day, "not available", 0.0
    t0 = time.time()
    last_err = ""
    for attempt in range(TRIES):
        for method, getter in [("fast", get_fast), ("standard", get_standard)]:
            try:
                sub = getter(info, day)
                save_daily(sub, day, out_file)
                return day, f"ok ({method})", time.time() - t0
            except Exception as e:
                last_err = f"{method}: {type(e).__name__}: {str(e)[:80]}"
        time.sleep(5 * (attempt + 1))
    return day, "failed (" + last_err + ")", time.time() - t0


if TEST_MODE and URLS:
    check_day = sorted(URLS)[-1]
    try:
        t0 = time.time()
        a = get_fast(URLS[check_day], check_day)
        t_fast = time.time() - t0
        t0 = time.time()
        b = get_standard(URLS[check_day], check_day)
        t_std = time.time() - t0
        diff = max(float(np.nanmax(np.abs(a[v].values - b[v].values))) / max(float(np.nanmax(np.abs(b[v].values))), 1e-30) for v in VARIABLES)
        print(f"Check on {check_day:%Y-%m-%d}: fast {t_fast:.0f} s, standard {t_std:.0f} s, largest relative difference {diff:.1e} ({'identical' if diff < 1e-5 else 'DIFFERENT: tell Claude'})")
    except Exception as e:
        print("Fast method not available:", type(e).__name__, str(e)[:150], "-> the standard method will be used")


run_days = [d for d in todo if d in URLS]
if TEST_MODE:
    run_days = run_days[:TEST_DAYS]
print(f"Fetching {len(run_days)} days with {WORKERS} at a time ...")
t_start = time.time()
results = []
with cf.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for k, (day, status, secs) in enumerate(pool.map(fetch_day, run_days), start=1):
        results.append((day, status, secs))
        if not status.startswith("ok") or k % 25 == 0 or TEST_MODE:
            print(f"{day:%Y-%m-%d}: {status} ({secs:.0f} s)   [{k}/{len(run_days)}, {time.time() - t_start:.0f} s so far]")
elapsed = time.time() - t_start
ok = [r for r in results if r[1].startswith("ok")]
failed = [r for r in results if r[1].startswith("failed")]
print(f"Done: {len(ok)} ok, {len(failed)} failed, in {elapsed:.0f} s")
if ok:
    per_day = elapsed / len(ok)
    remaining = len([d for d in days if not (OUT_DIR / f"merra2_{d:%Y%m%d}.nc").exists()])
    methods = pd.Series([r[1] for r in ok]).value_counts().to_dict()
    print(f"Methods used: {methods}")
    print(f"Average {per_day:.1f} s per day with {WORKERS} at a time; the remaining {remaining} days would take about {remaining * per_day / 60:.0f} minutes")
if failed:
    print("Re-run this cell to retry the failed days.")

Fetching 2166 days with 5 at a time ...
2003-09-06: ok (fast) (8 s)   [25/2166, 44 s so far]
2003-10-01: ok (fast) (9 s)   [50/2166, 88 s so far]
2003-10-26: ok (fast) (9 s)   [75/2166, 132 s so far]
2004-08-20: ok (fast) (9 s)   [100/2166, 174 s so far]
2004-09-14: ok (fast) (9 s)   [125/2166, 219 s so far]
2004-10-09: ok (fast) (9 s)   [150/2166, 264 s so far]
2005-08-03: ok (fast) (8 s)   [175/2166, 309 s so far]
2005-08-28: ok (fast) (8 s)   [200/2166, 350 s so far]
2005-09-22: ok (fast) (9 s)   [225/2166, 394 s so far]
2005-10-17: ok (fast) (9 s)   [250/2166, 438 s so far]
2006-08-11: ok (fast) (10 s)   [275/2166, 484 s so far]
2006-09-05: ok (fast) (9 s)   [300/2166, 528 s so far]
2006-09-30: ok (fast) (9 s)   [325/2166, 572 s so far]
2006-10-25: ok (fast) (8 s)   [350/2166, 616 s so far]
2007-08-19: ok (fast) (9 s)   [375/2166, 660 s so far]
2007-09-13: ok (fast) (9 s)   [400/2166, 706 s so far]
2007-10-08: ok (fast) (9 s)   [425/2166, 753 s so far]
2008-08-02: ok (fast) (9 s)  

## Section 3: check what is there

In [11]:
files = sorted(OUT_DIR.glob("merra2_*.nc"))
have = pd.DatetimeIndex([pd.Timestamp(f.stem.split("_")[1]) for f in files])
per_year = pd.Series(1, index=have).groupby(have.year).sum().reindex(range(FIRST_YEAR, LAST_YEAR + 1), fill_value=0)
print("Days downloaded per season (92 = complete):")
print(per_year.to_string())
print(f"Total {len(files)} files, {sum(f.stat().st_size for f in files) / 1e6:.0f} MB")
if files:
    x = xr.open_dataset(files[-1])
    print("Last file:", files[-1].name, dict(x.sizes), list(x.data_vars))

Days downloaded per season (92 = complete):
2003    92
2004    92
2005    92
2006    92
2007    92
2008    92
2009    92
2010    92
2011    92
2012    92
2013    92
2014    92
2015    92
2016    92
2017    92
2018    92
2019    92
2020    92
2021    92
2022    92
2023    92
2024    92
2025    92
2026    62
Total 2178 files, 1217 MB
Last file: merra2_20261001.nc {'time': 1, 'lat': 91, 'lon': 113} ['OCSMASS', 'BCSMASS', 'SSSMASS25', 'DUSMASS25', 'SO4SMASS', 'TOTEXTTAU', 'OCCMASS', 'BCCMASS', 'OCFLUXU', 'OCFLUXV', 'BCFLUXU', 'BCFLUXV']


___
**END**